# Разведочный анализ данных (Explanatory Data Analysis, EDA) в Python

#### Зачем нужен разведочный анализ как первый этап решения задачи машинного обучения:
- Понимание структуры, распределений и особенностей данных
- Выявление аномалий, ошибок и пропусков
- Выявление взаимосвязей между признаками
- Выбор признаков
- Понимание необходимости порождения новых признаков на основе существующих (feature engeneering)



#### Полезные ссылки:
- <a href="https://pandas.pydata.org/">Документация Pandas</a>
- <a href="https://matplotlib.org/">Документация Matplotlib</a>
- <a href="https://seaborn.pydata.org/">Документация Seaborn</a>

#### Дополнительные материалы:

- Маккинни У. Python и анализ данных / Пер. с англ. Слинкин А.А. – М: ДМК Пресс, 2015. – 482 с. ([Google Books]
- Брюс П., Брюс Э. Практическая статистика для специалистов Data Science. – БХВ-Петербург, 2018. — 304 с. ([Google Books](https://books.google.ru/books?hl=ru&lr=&id=l_6MDwAAQBAJ&oi=fnd&pg=PA5&dq=%D0%BF%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F+%D1%81%D1%82%D0%B0%D1%82%D0%B8%D1%81%D1%82%D0%B8%D0%BA%D0%B0+%D0%B4%D0%BB%D1%8F+%D1%81%D0%BF%D0%B5%D1%86%D0%B8%D0%B0%D0%BB%D0%B8%D1%81%D1%82%D0%BE%D0%B2+data+science&ots=fB2sdc0NnS&sig=S7_kC8Nv2Ipg5By2UbTDVDGVvqE&redir_esc=y#v=onepage&q=%D0%BF%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F%20%D1%81%D1%82%D0%B0%D1%82%D0%B8%D1%81%D1%82%D0%B8%D0%BA%D0%B0%20%D0%B4%D0%BB%D1%8F%20%D1%81%D0%BF%D0%B5%D1%86%D0%B8%D0%B0%D0%BB%D0%B8%D1%81%D1%82%D0%BE%D0%B2%20data%20science&f=false))
- [Лекции с YouTube](https://www.youtube.com/watch?v=Yhrp-YVZH84)
- [Еще лекции с YouTube](https://youtu.be/uJpDzHGUamg?si=29xzZdo2AzE12nOH)


In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns


## Загружаем данные

Будем работать с данными метеостанции Малые Кармакулы, взятыми из [архива ВНИИГМИ-МЦД](http://meteo.ru/data/)

In [ ]:
data_path = 'EDA_demo_data/wr282025.txt'
fields_path = 'EDA_demo_data/fld282025a0.txt'

# Загружаем информацию о полях из файла данных ВНИИГМИ-МЦД c фиксированной шириной колонок
df_fields = pd.read_fwf (fields_path, encoding = "cp1251", header = None)

display (df_fields)


In [ ]:

df_fields[3] = [' '.join(field.split()) for field in df_fields[3]]
display (df_fields[3])


In [ ]:
df = pd.read_csv (data_path, sep=';', header=None, names=df_fields[3])
df.head()


#### Подготовка данных 

In [ ]:
df['datetime'] = [pd.Timestamp(year,month,day,hour) for year,month,day,hour in zip(df['Год по Гринвичу'], df['Месяц по Гринвичу'], df['День по Гринвичу'], df['Срок по Гринвичу'])]

df = df.drop (columns=[x for x in df.columns if 'по Гринвичу' in x or 'Синоптический индекс' in x])

df = df.set_index ('datetime')

new_columns = {'Общее количество облачности': 'tcc', 
               'Количество облачности нижнего яруса': 'lcc', 
               'Направление ветра': 'wdir', 
               'Средняя скорость ветра': 'wvel', 
               'Максимальная скорость ветра': 'wgust',
               'Сумма осадков': 'prec', 
               'Температура поверхности почвы': 'tg', 
               'Температура воздуха по сухому терм-ру': 'ta', 
               'Относительная влажность воздуха': 'rh', 
               'Температура точки росы': 'td', 
               'Атмосферное давление на уровне станции': 'ps', 
               'Атмосферное давление на уровне моря': 'psl'}

df = df.rename(columns=new_columns)
display(df.head())


### Изучим типы данных

In [ ]:
display(df.info())



In [ ]:
# Преобразуем текстовые значения в числовые
df = df.apply(pd.to_numeric, errors='coerce')
display(df.info())

# Сохраним результаты в новый csv-файл
df.to_csv('EDA_demo_data/karmakuly.csv')


## Визуализация временных рядов

Посмотрим на скорость ветра для эпизора боры 10–12 Декабря 2006 г. из статьи [(Shestakova and Debolskiy, 2022)](https://www.mdpi.com/2073-4433/13/7/1108)

In [ ]:
#%matplotlib inline 
%matplotlib qt
plt.rcParams['figure.dpi'] = 300

plt.figure()
df['09-Dec-2006':'13-Dec-2006'][['wvel', 'wgust']].plot(ax = plt.gca())

plt.figure()
df[['wvel', 'wgust']].plot(ax = plt.gca())
plt.xlim(('09-Dec-2006','13-Dec-2006'))


Теперь интерактивный график с использованием Plotly

In [ ]:
pd.options.plotting.backend = "plotly"

fig = df['2006':'2007'][['wvel', 'wgust']].plot()

fig.update_layout(xaxis_range=['2006-12-09', '2006-12-13'])
fig.show()

pd.options.plotting.backend = "matplotlib"


## Доступ к столбцам и колонкам в DataFrame

In [ ]:
# Доступ по индексу
df['2010-01-01 03':'2010-01-01 03']


In [ ]:
# Доступ по порядковому номеру
df[445:448]

In [ ]:
# Доступ по массиву булевских значений
df[df['wgust'] > 30]

In [ ]:
# Доступ к колонкам по имени
df[['wdir', 'wvel']].head()

In [ ]:
# Одновременная обрезка по колонкам и столбцам

#df.loc['2010-01-01 03']
#df.loc['2010-01-01 03':'2010-01-01 12', ['wdir', 'wvel']]
df.iloc[3333, 4:7]

## Анализ доступности данных

In [ ]:
%matplotlib inline

data_missing = df.isnull ().astype(float).resample('M', label='left').mean()

#display(data_missing)

plt.figure()
plt.pcolormesh(data_missing.index, data_missing.columns, data_missing.T.to_numpy())
_ = plt.xticks(rotation=45)

#sys.exit()


# Одномерный разведочный анализ данных (univariate analysis)

### Анализ статистических моментов 

In [ ]:
#df['wvel'].median()
#df.describe()
df.describe().T #Транспонирование

### Анализ уникальных значений

In [ ]:
nunique = lambda x: len (pd.unique(x))

print ('Уникальные значения в столбцах: ')
display(df.apply (pd.unique))

print ('Число уникальных значений в столбцах:  ')
display(df.nunique())

In [ ]:
# Уникальные значения балла облачности
np.sort (df['tcc'].unique())

In [ ]:
# Подсчет уникальных значений

plt.rcParams['figure.dpi'] = 200

#df_cr = df['2015':]

display(df['tcc'].value_counts().sort_index())
plt.figure()
df['tcc'].value_counts().sort_index().plot.bar() #
plt.figure()
df['tcc'].value_counts().sort_index().plot.pie() #

### Анализ функций распределения

In [ ]:
#Гистограммы распределения
%matplotlib inline
df.hist(figsize=(20,20), density=True)

### Что с направлением ветра?

In [ ]:
df[df['wdir'] == 999]

In [ ]:
mask_999 = df['wdir'] == 999

# Создаем маски для соседних строк
mask_prev = mask_999.shift(1, fill_value=False)  # предыдущие строки
mask_next = mask_999.shift(-1, fill_value=False)  # следующие строки

# Объединяем все маски
combined_mask = mask_999 | mask_prev | mask_next

result = df[combined_mask]
display(result)

### Графики ядерной плотности (kernel density plots)

Ядерные оценки плотности — KDE, Kernel Density Estimates — способ что-то понять о распределении, когда неизвестно ничего. Такого рода методы называют непараметрическими, они иллюстрируют разницу подходов в статистике и теории вероятностей: если в теории вероятности известно распределение и исследуются его свойства, то в статистике зачастую известны только данные, и по их свойствам угадывается распределение.

Смысл: в каждую точку выборки поставили отмасштабированное ядро так, будто эта точка — центр ядра, а затем усреднили значения соседних точек с весами, заданными этим ядром. Вместо тысячи слов — [интерактивная иллюстрация](https://mathisonian.github.io/kde/).

In [ ]:
#Гистограммы и графики ядерной плотности 

plt.figure()
df[['wvel', 'wgust']].plot.hist(alpha=0.5, density = True, bins=np.arange(-0.5, 40.5))
df[['wvel', 'wgust']].plot.density(ax = plt.gca(), color = ['blue', 'red'], bw_method=0.5) 
plt.xlim(0, 40)


In [ ]:
decades = np.arange(1970, 2010, 10)

for decade_start in decades:
    df4decade = df[str(decade_start) + '-01-01':str(decade_start + 10) + '-12-31']

    plt.figure(figsize=(8,4))
    plt.subplot(1,2,1)
    df4decade['wvel'].value_counts(normalize = True).sort_index().plot.bar(ax = plt.gca(), color = 'blue', alpha = 0.5, label = 'wvel')
    plt.xlim(0, 30)
    plt.subplot(1,2,2)
    df4decade['wgust'].value_counts(normalize = True).sort_index().plot.bar(ax = plt.gca(), color= 'red', alpha = 0.5, label = 'wgust')
    plt.suptitle(f'{decade_start} - {decade_start + 10}')
    plt.xlim(0, 30)

### Ищучим динамику повторяемости подозрительных значений во времени

In [ ]:
counts = df.groupby (df.index.year)['wgust'].value_counts(normalize = True)

years = counts.index.get_level_values(0).unique()
vals  = counts.index.get_level_values(1).unique() 

counts = pd.concat([pd.DataFrame (counts[year]).T.reindex(sorted(vals), axis=1) for year in years]).fillna(0)
counts['year'] = years
counts = counts.set_index ('year')
counts.loc[:, 13:15].plot()



### Идентификация выбросов 

In [ ]:
# Ящики с усами
#df.plot.box()
df[['wvel', 'wgust']].plot.box()

In [ ]:
df.plot(kind='box', subplots=True,  figsize=(15, 5))
plt.subplots_adjust(wspace=0.9) 

## Двумерный анализ данных (bivariate data analysis)

#### Корреляция

In [ ]:
corrs = df.corr()

display (corrs)

In [ ]:


s = sns.heatmap (corrs, annot=True, annot_kws={'fontsize': 'xx-small'})
#plt.set(s,  fontsize = 'x-small')

### Диаграммы рассеяния

In [ ]:
plt.figure()
plt.scatter (df['ta'], df['td'])
#plt.scatter (df['wvel'], df['wgust'])
y_lim = plt.ylim()
display(y_lim)
plt.plot(y_lim, y_lim, '-k')
plt.gca().set_aspect('equal')

### Шестиугольные корзины

In [ ]:
#plt.hexbin(df['wvel'], df['wgust'], gridsize=50, mincnt = 2)

plt.hexbin(df['ta'], df['td'], gridsize=50, mincnt = 1)
plt.gca().set_aspect('equal')
plt.grid()
plt.colorbar()
# plt.xlim((0, 50))
# plt.ylim((0, 50))

In [ ]:
def month_to_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'


df['season'] = [month_to_season (m) for m in df.index.month] 

sns.jointplot(df, x = 'ta', y = 'td', hue = 'season', palette="tab10") #, kind = 'hex', color = 'blue') palette="tab10") #
plt.gca().set_aspect('equal')
plt.grid()

In [ ]:
# Ящики с усами для двумерного анализа
sns.boxplot(df, x = 'tcc', y = 'rh')

In [ ]:
sns.pairplot (df[['ps', 'psl']])

#### Многомерный анализ данных

In [ ]:
bins = np.arange (-30,30,10) 
labels = [f'{bins[i]} - {bins[i+1]}' for i in range (len(bins)-1)]
df['ta_range'] = pd.cut(df['ta'], bins=bins, labels=labels, right=True)


g = sns.FacetGrid(df, col = 'ta_range', height=4, aspect = 1)  

g.map(plt.scatter, 'ps', 'psl') #, gridsize=20, mincnt = 1)
